# Contextual Compression: Refining Retrieval Output for Optimal LLM Context

Retrieval-Augmented Generation (RAG) systems are powerful, but their effectiveness is often limited by the quality of the retrieved context. A standard vector search retrieves documents based purely on semantic similarity, which can sometimes pull in tangential or overly broad information—a phenomenon known as "context noise." This noise dilutes the signal and forces the Large Language Model (LLM) to spend valuable tokens processing irrelevant details, potentially leading to hallucination or diminished answer quality.

Contextual Compression addresses this critical bottleneck by introducing a refinement layer *after* initial retrieval but *before* context injection into the prompt. Instead of simply passing all retrieved chunks, compression techniques intelligently filter, summarize, and extract only the most relevant passages that directly address the query's intent. We explore advanced methods like using LLMs to extract key information (LLMChainExtractor) or applying specialized filters (EmbeddingsFilter) to ensure the final context is maximally dense with actionable knowledge.

Mastering contextual compression is essential for building production-grade, robust RAG pipelines. By learning how to refine retrieved documents, you move beyond basic semantic search and build systems that are highly focused, efficient, and capable of providing precise answers even when the underlying corpus is vast and noisy. This skill set is crucial for advanced orchestration frameworks like LangGraph, where context quality directly dictates the success of subsequent reasoning steps.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Understand Context Noise:** Identify the limitations of basic vector retrieval when dealing with large or complex document sets.
*   **Implement Compression Strategies:** Utilize `ContextualCompressionRetriever` to wrap and enhance standard retrievers.
*   **Apply Specific Compressors:** Demonstrate how different compressors, such as `LLMChainExtractor` and `EmbeddingsFilter`, can be used to refine retrieved documents based on specific criteria (e.g., relevance or embedding similarity).
*   **Optimize RAG Context:** Design a multi-stage retrieval pipeline that ensures the LLM receives only the most focused and actionable context, thereby improving answer accuracy and reducing token usage.


### Setup and Imports for Advanced Retrieval

This cell imports necessary components for advanced retrieval-augmented generation (RAG). It brings in tools like `Document` for data handling, `InMemoryVectorStore` for local vector storage, OpenAI models (`OpenAIEmbeddings`, `ChatOpenAI`), and crucially, specialized classes from `langchain_classic.retrievers` such as `ContextualCompressionRetriever` and various document compressors (e.g., `LLMChainExtractor`) used to refine retrieved context.


In [ ]:
from dotenv import load_dotenv
# Load environment variables (like API keys) from a .env file

from langchain_core.documents import Document
# Import the core Document class for handling text chunks

from langchain_core.vectorstores import InMemoryVectorStore
# Import an in-memory vector store for demonstration/testing

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# Import embedding and chat models from OpenAI

from langchain_classic.retrievers import ContextualCompressionRetriever
# Import the main retriever class that applies context compression

from langchain_classic.retrievers.document_compressors import (
    LLMChainExtractor, # Compressor using LLMs to extract key information from chains
    EmbeddingsFilter, # Filter based on embedding similarity/criteria
    DocumentCompressorPipeline, # Pipeline for managing multiple compression steps
)
# Import various document compressor types used within the ContextualCompressionRetriever


In [5]:
load_dotenv()

True

### Initialization of Core Components

This cell initializes the essential components for our RAG pipeline: an embedding model and a Large Language Model (LLM). `OpenAIEmbeddings` converts text into numerical vectors, while `ChatOpenAI` provides the generative AI capabilities needed for reasoning and synthesis.


In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small") # Initializes the embedding model used to convert text chunks into high-dimensional vector representations.
llm = ChatOpenAI(model="gpt-5-mini", temperature=0) # Initializes the chat-based LLM, setting temperature to 0 for deterministic and reliable outputs.


### Data Initialization and Setup

This cell initializes a list of `Document` objects. Each document simulates a chunk of raw text data covering diverse, complex topics (AI, climate change, space, etc.). This setup is crucial for demonstrating advanced RAG techniques like contextual compression, as it provides varied source material with mixed relevance to test the system's ability to extract core information.


In [6]:
# Dummy documents covering different topics, each with a mix of relevant and tangential info
docs = [
    Document(
        page_content=(
            "Artificial intelligence has made remarkable strides in natural language processing, "
            "with large language models now capable of generating human-quality text and code. "
            "Computer vision systems can identify objects in images with superhuman accuracy, "
            "powering applications from autonomous vehicles to medical imaging diagnostics. "
            "However, the rapid advancement of AI has raised significant ethical concerns about "
            "job displacement, algorithmic bias, and the concentration of power among a few tech companies."
        ),
        metadata={"topic": "artificial_intelligence"},
    ),
    Document(
        page_content=(
            "Global temperatures have risen by approximately 1.1 degrees Celsius since pre-industrial "
            "times, driven primarily by the burning of fossil fuels. The melting of polar ice caps has "
            "accelerated, contributing to rising sea levels that threaten coastal communities worldwide. "
            "Renewable energy adoption is growing rapidly, with solar and wind power becoming cheaper "
            "than coal in many regions. Governments are implementing carbon pricing mechanisms and "
            "investing in green infrastructure to meet Paris Agreement targets."
        ),
        metadata={"topic": "climate_change"},
    ),
    Document(
        page_content=(
            "NASA's Artemis program aims to return humans to the Moon by the mid-2020s, establishing "
            "a sustainable presence as a stepping stone to Mars. Private companies like SpaceX are "
            "developing reusable rocket technology that has dramatically reduced launch costs. "
            "The James Webb Space Telescope has captured unprecedented images of distant galaxies, "
            "revealing new insights about the early universe. Asteroid mining is being explored as a "
            "potential source of rare minerals needed for electronics manufacturing."
        ),
        metadata={"topic": "space_exploration"},
    ),
    Document(
        page_content=(
            "CRISPR gene editing technology has revolutionized medical genomics, enabling precise "
            "modifications to DNA sequences that were previously impossible. Researchers are using "
            "genomic data to develop personalized medicine approaches, tailoring treatments based on "
            "an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by "
            "COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare "
            "genetic disorders. Hospital information systems are increasingly integrating genomic data "
            "to support clinical decision-making at the point of care."
        ),
        metadata={"topic": "medicine"},
    ),
    Document(
        page_content=(
            "The global economy is navigating a period of high inflation driven by supply chain "
            "disruptions, energy price volatility, and post-pandemic demand surges. Central banks "
            "worldwide have raised interest rates aggressively to combat inflation, impacting housing "
            "markets and consumer spending. Cryptocurrency regulation is becoming a priority for "
            "financial authorities, with the EU's MiCA framework setting a global precedent. "
            "Trade tensions between major economies continue to reshape global supply chains, "
            "pushing companies toward nearshoring and diversification strategies."
        ),
        metadata={"topic": "economics"},
    ),
    Document(
        page_content=(
            "Quantum computing has reached a critical milestone with several companies demonstrating "
            "quantum advantage on specific computational tasks. Error correction remains the biggest "
            "challenge, as current quantum processors are highly susceptible to noise and decoherence. "
            "Quantum simulation of molecular structures could transform drug discovery by accurately "
            "modeling protein folding and chemical interactions. Major tech companies and governments "
            "are investing billions in quantum research, viewing it as essential for national security "
            "and economic competitiveness."
        ),
        metadata={"topic": "quantum_computing"},
    ),
]

print(f"Created {len(docs)} documents")



Created 6 documents


In [8]:
print(docs[0].page_content)

Artificial intelligence has made remarkable strides in natural language processing, with large language models now capable of generating human-quality text and code. Computer vision systems can identify objects in images with superhuman accuracy, powering applications from autonomous vehicles to medical imaging diagnostics. However, the rapid advancement of AI has raised significant ethical concerns about job displacement, algorithmic bias, and the concentration of power among a few tech companies.


### Vector Store Initialization and Base Retriever Setup

This cell initializes an in-memory vector store using the provided documents (`docs`) and embeddings. It then creates a `base_retriever` object from this store, configured to retrieve the top 3 most relevant documents for any given query.


In [9]:
# Build a vector store and base retriever that returns top 3 docs

# Initialize an in-memory vector store using the loaded documents and embeddings.
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)

# Create the base retriever object from the vector store, specifying 'k=3' to retrieve the top 3 relevant documents.
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


### Baseline Retrieval (No Compression)

This cell demonstrates the baseline retrieval process by invoking the `base_retriever` directly with the query. It prints the full content of every retrieved document, allowing us to see the raw context before any compression or filtering is applied.


In [14]:
# Baseline: retrieve without any compression to see full document content
query = "How is CRISPR acting as a big enabler in creating personalized medicine?"

# Invoke the base retriever using the query to get all initial results
base_results = base_retriever.invoke(query)

# Iterate through the retrieved documents and print their full content
for i, doc in enumerate(base_results):
    print(f"--- Result {i+1} [{doc.metadata.get('topic')}] ---")
    print(doc.page_content)
    print()


--- Result 1 [medicine] ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare genetic disorders. Hospital information systems are increasingly integrating genomic data to support clinical decision-making at the point of care.

--- Result 2 [quantum_computing] ---
Quantum computing has reached a critical milestone with several companies demonstrating quantum advantage on specific computational tasks. Error correction remains the biggest challenge, as current quantum processors are highly susceptible to noise and decoherence. Quantum simulation of molecular structures could transform drug discovery by accurately

### Contextual Compression Retrieval

This cell implements a sophisticated retrieval step using `ContextualCompressionRetriever`. It leverages an LLM-powered extractor (`LLMChainExtractor`) to intelligently compress the retrieved documents, ensuring that only the most relevant and contextually important snippets are passed to the final generation model. This significantly reduces noise and improves answer quality by focusing the LLM's attention.


In [28]:
# LLMChainExtractor: uses an LLM to extract only the relevant portions from each document
# Initializes the compressor using a specified LLM instance.
compressor = LLMChainExtractor.from_llm(llm)

# Creates the ContextualCompressionRetriever, combining the base retriever (source) 
# with the newly created LLM-based compressor.
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

# Executes the retrieval process, which now includes the compression step.
compressed_results = compression_retriever.invoke(query)

# Iterates through and prints the compressed results for inspection.
for i, doc in enumerate(compressed_results):
    print(f"--- Compressed Result {i+1} [{doc.metadata.get('topic')}] ---")
    print(doc.page_content)
    print()



--- Compressed Result 1 [medicine] ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile.



### Contextual Compression Filtering

This cell implements a sophisticated filtering mechanism using `EmbeddingsFilter` and wraps it within a `ContextualCompressionRetriever`. The `EmbeddingsFilter` ensures that only documents whose embedding similarity to the query exceeds a specified threshold (0.732) are considered, significantly improving retrieval precision by dropping irrelevant chunks before they reach the final LLM stage.


In [26]:
# EmbeddingsFilter: drops entire documents whose embedding similarity to the query is below the threshold
embeddings_filter = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=0.732,
)

# Wrap the base retriever with the embeddings filter for contextual compression
compression_retriever_emb = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever,
)

# Invoke the compressed retriever with the query to get filtered results
emb_results = compression_retriever_emb.invoke(query)

# Iterate through and print the retrieved, filtered documents
for i, doc in enumerate(emb_results):
    # Retrieve the relevance score from metadata for display
    score = doc.metadata.get("relevance_score", "N/A")
    print(f"--- Filtered Result {i+1} [{doc.metadata.get('topic')}] (score: {score}) ---")
    # Print the actual content of the document chunk
    print(doc.page_content)
    print()



--- Filtered Result 1 [medicine] (score: N/A) ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare genetic disorders. Hospital information systems are increasingly integrating genomic data to support clinical decision-making at the point of care.



This cell executes the advanced contextual compression pipeline. It first uses a `DocumentCompressorPipeline` to chain multiple filtering and extraction steps (embeddings filter $\rightarrow$ LLM compressor), minimizing expensive LLM calls. This compressed output is then used by the `ContextualCompressionRetriever` to retrieve highly relevant, contextually optimized documents for the given query.


In [27]:
# DocumentCompressorPipeline: chain multiple compressors together
# This pipeline defines a sequence of compression steps.
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[embeddings_filter, compressor]
)

# ContextualCompressionRetriever uses the defined pipeline to refine retrieval.
# It takes the base retriever and applies the advanced compression logic.
compression_retriever_pipeline = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=base_retriever,
)

# Execute the pipeline with the query, retrieving compressed results.
pipeline_results = compression_retriever_pipeline.invoke(query)

# Iterate and print the content of each retrieved document for inspection.
for i, doc in enumerate(pipeline_results):
    print(f"--- Pipeline Result {i+1} [{doc.metadata.get('topic')}] ---")
    print(doc.page_content)
    print()



--- Pipeline Result 1 [medicine] ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile.

